# Modélisation — Système de recommandation
Deux approches : Content-Based Filtering et Collaborative Filtering (SVD).

In [ ]:
import pandas as pd
import numpy as np
import pickle
import zipfile
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix

DATA_DIR = '../data'
N_RECO = 5

In [ ]:
meta = pd.read_csv(f'{DATA_DIR}/articles_metadata.csv')

with zipfile.ZipFile(f'{DATA_DIR}/clicks.zip', 'r') as z:
    csv_files = [f for f in z.namelist() if f.endswith('.csv')]
    clicks = pd.concat(
        [pd.read_csv(z.open(f)) for f in csv_files],
        ignore_index=True
    )

# Cast explicite en int (le concat de 385 fichiers produit du object)
clicks['user_id'] = clicks['user_id'].astype(int)
clicks['click_article_id'] = clicks['click_article_id'].astype(int)

with open(f'{DATA_DIR}/articles_embeddings.pickle', 'rb') as f:
    embeddings = pickle.load(f)

print(f'Articles  : {len(meta)}')
print(f'Clicks    : {len(clicks)}')
print(f'Embeddings: {embeddings.shape}')


## 2. Préparation

In [ ]:
# Encoder user_id et article_id en index continus
user_ids = clicks['user_id'].unique()
article_ids = clicks['click_article_id'].unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
article2idx = {a: i for i, a in enumerate(article_ids)}
idx2article = {i: a for a, i in article2idx.items()}

print(f'Users    : {len(user2idx)}')
print(f'Articles : {len(article2idx)}')

In [ ]:
# Matrice user-article sparse (nombre de clics)
rows = clicks['user_id'].map(user2idx)
cols = clicks['click_article_id'].map(article2idx)
data = np.ones(len(clicks))

user_article_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(user2idx), len(article2idx))
)
print(f'Matrice user-article : {user_article_matrix.shape}')
print(f'Sparsité             : {1 - user_article_matrix.nnz / (user_article_matrix.shape[0] * user_article_matrix.shape[1]):.4%}')

In [ ]:
# ACP sur les embeddings (250 → 50 dimensions)
N_COMPONENTS = 50
pca = PCA(n_components=N_COMPONENTS, random_state=42)
embeddings_reduced = pca.fit_transform(embeddings)
variance_explained = pca.explained_variance_ratio_.sum()

print(f'Embeddings réduits : {embeddings_reduced.shape}')
print(f'Variance expliquée : {variance_explained:.2%}')

## 3. Content-Based Filtering
Pour un user donné : moyenne des embeddings de ses articles → similarité cosinus avec tous les articles.

In [ ]:
def get_user_profile(user_id, clicks_df, embeddings_arr):
    """Calcule le profil d'un user = moyenne des embeddings de ses articles cliqués."""
    user_articles = clicks_df[clicks_df['user_id'] == user_id]['click_article_id'].values
    if len(user_articles) == 0:
        return None
    valid = user_articles[user_articles < len(embeddings_arr)]
    return embeddings_arr[valid].mean(axis=0)

In [ ]:
def recommend_content_based(user_id, clicks_df, embeddings_arr, n=N_RECO):
    """Retourne les N articles les plus similaires au profil du user."""
    profile = get_user_profile(user_id, clicks_df, embeddings_arr)
    if profile is None:
        # Cold start : retourner les articles les plus populaires
        return clicks_df['click_article_id'].value_counts().head(n).index.tolist()

    already_clicked = set(clicks_df[clicks_df['user_id'] == user_id]['click_article_id'].values)

    scores = cosine_similarity([profile], embeddings_arr)[0]
    top_indices = np.argsort(scores)[::-1]

    recommendations = []
    for idx in top_indices:
        if idx not in already_clicked:
            recommendations.append(idx)
        if len(recommendations) == n:
            break

    return recommendations

In [ ]:
# Test sur un user
test_user = clicks['user_id'].iloc[0]
reco_cb = recommend_content_based(test_user, clicks, embeddings_reduced)

print(f'User : {test_user}')
print(f'Articles cliqués   : {clicks[clicks["user_id"] == test_user]["click_article_id"].tolist()}')
print(f'Recommandations CB : {reco_cb}')

## 4. Collaborative Filtering (SVD)
Décomposition matricielle : on factorise la matrice user-article pour prédire les clics manquants.

In [ ]:
# SVD sur la matrice user-article
N_FACTORS = 50
svd = TruncatedSVD(n_components=N_FACTORS, random_state=42)
user_factors = svd.fit_transform(user_article_matrix)
item_factors = svd.components_.T

print(f'User factors  : {user_factors.shape}')
print(f'Item factors  : {item_factors.shape}')
print(f'Variance expliquée : {svd.explained_variance_ratio_.sum():.2%}')

In [ ]:
def recommend_cf(user_id, user2idx, user_factors, item_factors, idx2article, clicks_df, n=N_RECO):
    """Retourne les N articles avec le meilleur score SVD pour le user."""
    if user_id not in user2idx:
        # Cold start : articles populaires
        return clicks_df['click_article_id'].value_counts().head(n).index.tolist()

    user_idx = user2idx[user_id]
    scores = user_factors[user_idx] @ item_factors.T

    already_clicked = set(clicks_df[clicks_df['user_id'] == user_id]['click_article_id'].values)
    already_clicked_idx = {article2idx[a] for a in already_clicked if a in article2idx}

    top_indices = np.argsort(scores)[::-1]
    recommendations = []
    for idx in top_indices:
        if idx not in already_clicked_idx:
            recommendations.append(idx2article[idx])
        if len(recommendations) == n:
            break

    return recommendations

In [ ]:
# Test sur le même user
reco_cf = recommend_cf(test_user, user2idx, user_factors, item_factors, idx2article, clicks)

print(f'User : {test_user}')
print(f'Articles cliqués   : {clicks[clicks["user_id"] == test_user]["click_article_id"].tolist()}')
print(f'Recommandations CF : {reco_cf}')

## 5. Modèle hybride
Si l'user est connu et a suffisamment de clics → CF. Sinon → Content-Based.

In [ ]:
MIN_CLICKS_FOR_CF = 3

def recommend_hybrid(user_id, clicks_df, embeddings_arr, user2idx, user_factors, item_factors, idx2article, n=N_RECO):
    user_clicks = clicks_df[clicks_df['user_id'] == user_id]
    n_clicks = len(user_clicks)

    if user_id in user2idx and n_clicks >= MIN_CLICKS_FOR_CF:
        method = 'CF'
        reco = recommend_cf(user_id, user2idx, user_factors, item_factors, idx2article, clicks_df, n)
    else:
        method = 'Content-Based'
        reco = recommend_content_based(user_id, clicks_df, embeddings_arr, n)

    return reco, method

In [ ]:
# Test sur plusieurs users
test_users = clicks['user_id'].value_counts().head(3).index.tolist()
test_users.append(99999999)  # user inconnu (cold start)

for uid in test_users:
    reco, method = recommend_hybrid(uid, clicks, embeddings_reduced, user2idx, user_factors, item_factors, idx2article)
    n_clicks = len(clicks[clicks['user_id'] == uid])
    print(f'User {uid} ({n_clicks} clics) → [{method}] {reco}')

## 6. Sauvegarde des modèles

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

# Sauvegarder tout ce dont l'Azure Function aura besoin
artifacts = {
    'embeddings_reduced': embeddings_reduced,
    'user_factors': user_factors,
    'item_factors': item_factors,
    'user2idx': user2idx,
    'article2idx': article2idx,
    'idx2article': idx2article,
    'clicks_per_user': clicks.groupby('user_id')['click_article_id'].apply(list).to_dict(),
    'popular_articles': clicks['click_article_id'].value_counts().head(20).index.tolist(),
}

with open('../models/recommendation_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

size_mb = os.path.getsize('../models/recommendation_artifacts.pkl') / 1024**2
print(f'Artefacts sauvegardés — {size_mb:.1f} MB')

In [ ]:
# Vérification : recharger et tester
with open('../models/recommendation_artifacts.pkl', 'rb') as f:
    loaded = pickle.load(f)

print('Clés disponibles :', list(loaded.keys()))
print('Test rechargement OK ✓')